# Agents的初始化脚本

每个冷启动的会话都需要交一次税。agent读相同的文件、尝试相同的探测、然后发现同样的路径。一个初始化脚本付一次税，然后把结果写到状态中。

## 问题描述

打开一个会话。agents猜Python版本，猜测试命令。遍历根仓库几次来找到入口。尝试导入包。问用户配置文件在哪。在做出一次真正的编辑之前，成千上万的token已经被消耗掉了，而这些通过一个脚本就可以完成。

修复方案是在agent做任何事情之前跑一个初始化脚本，然后写一份`init_report.json`，agent在启动的时候读它。

## 基本概念

```mermaid
flowchart TD
A[Session Start]-->B[init_agent.py] --> C[probe runtime/deps/paths/env/tests]--> D[init_report.json]-->E(healthy?)
E--yes-->F[Agent loop]
E--no-->G[fail loud, halt, surface to human]
```

### 初始化脚本检测什么

|检测项目|为什么重要|
|---|---|
|运行时版本|错误的Python版本导致静默版本bugs|
|依赖|在后续修复丢失的包比一开始贵好几倍|
|测试命令|agent必须知道怎么验证，如果命令没有，工作台就是损坏的|
|仓库路径|硬编码路径漂移，在一开始就处理并标记|
|环境变量|找不到模型API_KEY是失败表象|
|状态+面板新鲜|警惕崩溃会话带来的状态|
|上一个已知正常的提交|会话结束递交差异的锚点|

### 大声、快速地报告失败的地方

一个探测失败意味着停止并上浮给用户。不是“agent会自己想办法”，初始化最终要的是当工作台损坏的时候拒绝开始。

### 幂等（执行一次和执行多次结果一样）

初始化跑两次，第二次应该除了更新时间戳之外没有其他操作。

### 初始化和开启规则

规则描述的是行动的时候哪些是事实。初始化是构建这些规则可以被检查的环境所需的脚本。没有初始化的规则变成口头注意，没有规则的初始化带来抛光后的失败。

# 开始编码

对应本章核心：**冷启动税 → `init_agent` 探测 → `init_report.json`**、**不健康则 fail loud 拒进 loop**、**幂等重跑（只刷新时间戳）**、**初始化为规则检查铺环境**。  
玩具先跑通探测与门闩；生产段用 **LangChain 工具 + DeepSeek** 强制先 init。不硬凑 PyTorch。


## 1. 教学玩具：InitScript + init_report

探测：runtime / deps / test_command / paths / env / state 新鲜度 / last_known_good。  
关键项失败 → `WorkbenchBroken`；相同指纹再跑 → `rerun=true` 只更新 `generated_at`。


In [ ]:
from __future__ import annotations

import hashlib
import json
import os
import shutil
import sys
import tempfile
import time
from dataclasses import asdict, dataclass, field
from pathlib import Path
from typing import Any, Callable


REQUIRED_ENV = ("DEEPSEEK_API_KEY",)
REQUIRED_PATHS = ("src", "tests", "AGENTS.md", "agent_state.json", "task_board.json")


@dataclass
class ProbeResult:
    """单项探测结果。"""

    name: str
    ok: bool
    detail: str
    critical: bool = True


@dataclass
class InitReport:
    """初始化报告：agent 启动前必须读到的事实快照。"""

    healthy: bool
    fingerprint: str
    generated_at: float
    probes: list[dict[str, Any]] = field(default_factory=list)
    runtime: dict[str, Any] = field(default_factory=dict)
    test_command: str = ""
    last_known_good_commit: str = ""
    rerun: bool = False

    def to_dict(self) -> dict[str, Any]:
        """
        Returns:
            payload: 可落盘的 init_report.json。
        """
        return asdict(self)


class WorkbenchBroken(RuntimeError):
    """初始化探测失败：拒绝进入 agent loop。"""


def _sha(text: str) -> str:
    return hashlib.sha256(text.encode("utf-8")).hexdigest()[:16]


class InitScript:
    """会话冷启动税：探测一次，写入 init_report.json，幂等可重跑。"""

    def __init__(self, root: Path, *, required_env: tuple[str, ...] = REQUIRED_ENV) -> None:
        """
        Args:
            root: 仓库根。
            required_env: 必须存在的环境变量名。
        """
        self.root = Path(root)
        self.report_path = self.root / "init_report.json"
        self.required_env = required_env

    def probe_runtime(self) -> ProbeResult:
        """
        Returns:
            result: Python 主版本是否可用。
        """
        ver = sys.version_info
        ok = ver.major == 3 and ver.minor >= 10
        return ProbeResult(
            name="runtime",
            ok=ok,
            detail=f"python {ver.major}.{ver.minor}.{ver.micro}",
            critical=True,
        )

    def probe_deps(self) -> ProbeResult:
        """
        Returns:
            result: 关键依赖是否可 import。
        """
        missing: list[str] = []
        for mod in ("json", "pathlib"):
            try:
                __import__(mod)
            except ImportError:
                missing.append(mod)
        # 演示用：仓库内 requirements 标记的可选包
        req = self.root / "requirements.txt"
        if req.exists():
            for line in req.read_text(encoding="utf-8").splitlines():
                name = line.strip().split("==")[0].strip()
                if not name or name.startswith("#"):
                    continue
                try:
                    __import__(name.replace("-", "_"))
                except ImportError:
                    missing.append(name)
        return ProbeResult(
            name="deps",
            ok=not missing,
            detail="ok" if not missing else f"missing={missing}",
            critical=True,
        )

    def probe_test_command(self) -> ProbeResult:
        """
        Returns:
            result: 是否声明了可验证命令。
        """
        cfg = self.root / "workbench.json"
        cmd = ""
        if cfg.exists():
            cmd = str(json.loads(cfg.read_text(encoding="utf-8")).get("test_command", "")).strip()
        ok = bool(cmd)
        return ProbeResult(
            name="test_command",
            ok=ok,
            detail=cmd or "missing test_command in workbench.json",
            critical=True,
        )

    def probe_paths(self) -> ProbeResult:
        """
        Returns:
            result: 关键路径是否存在。
        """
        missing = [p for p in REQUIRED_PATHS if not (self.root / p).exists()]
        return ProbeResult(
            name="paths",
            ok=not missing,
            detail="ok" if not missing else f"missing={missing}",
            critical=True,
        )

    def probe_env(self) -> ProbeResult:
        """
        Returns:
            result: 关键环境变量是否已设置（不回显值）。
        """
        missing = [k for k in self.required_env if not os.getenv(k)]
        return ProbeResult(
            name="env",
            ok=not missing,
            detail="ok" if not missing else f"missing={missing}",
            critical=True,
        )

    def probe_state_fresh(self) -> ProbeResult:
        """
        Returns:
            result: agent_state / task_board 是否可读且不过期标记。
        """
        state_p = self.root / "agent_state.json"
        board_p = self.root / "task_board.json"
        if not state_p.exists() or not board_p.exists():
            return ProbeResult("state_fresh", False, "state or board missing", True)
        try:
            state = json.loads(state_p.read_text(encoding="utf-8"))
            board = json.loads(board_p.read_text(encoding="utf-8"))
        except json.JSONDecodeError as e:
            return ProbeResult("state_fresh", False, f"json broken: {e}", True)
        stale = bool(state.get("stale")) or bool(board.get("stale"))
        return ProbeResult(
            name="state_fresh",
            ok=not stale,
            detail="fresh" if not stale else "stale flag set (crash residue?)",
            critical=True,
        )

    def probe_last_good_commit(self) -> ProbeResult:
        """
        Returns:
            result: 是否记录了上一个已知正常提交锚点。
        """
        anchor = self.root / "last_known_good.txt"
        if not anchor.exists():
            return ProbeResult("last_known_good", False, "last_known_good.txt missing", True)
        sha = anchor.read_text(encoding="utf-8").strip()
        ok = bool(sha) and len(sha) >= 7
        return ProbeResult(
            name="last_known_good",
            ok=ok,
            detail=sha[:12] if ok else "empty anchor",
            critical=False,  # 可警告但不一定阻断（演示：critical=False 仍写入报告）
        )

    def _collect(self) -> list[ProbeResult]:
        return [
            self.probe_runtime(),
            self.probe_deps(),
            self.probe_test_command(),
            self.probe_paths(),
            self.probe_env(),
            self.probe_state_fresh(),
            self.probe_last_good_commit(),
        ]

    def fingerprint(self, probes: list[ProbeResult], test_command: str, commit: str) -> str:
        """
        Args:
            probes: 探测列表。
            test_command: 测试命令。
            commit: 锚点提交。
        Returns:
            fp: 环境指纹；幂等重跑时用于判断是否只需刷新时间戳。
        """
        blob = json.dumps(
            {
                "probes": [(p.name, p.ok, p.detail) for p in probes],
                "test_command": test_command,
                "commit": commit,
                "root": str(self.root.resolve()),
            },
            ensure_ascii=False,
            sort_keys=True,
        )
        return _sha(blob)

    def run(self, *, fail_loud: bool = True) -> InitReport:
        """
        执行初始化；若已有相同指纹的健康报告则只更新时间戳（幂等）。

        Args:
            fail_loud: 关键探测失败时是否抛 WorkbenchBroken。
        Returns:
            report: InitReport。
        """
        probes = self._collect()
        test_cmd = next((p.detail for p in probes if p.name == "test_command" and p.ok), "")
        commit = next((p.detail for p in probes if p.name == "last_known_good" and p.ok), "")
        fp = self.fingerprint(probes, test_cmd, commit)
        critical_ok = all(p.ok for p in probes if p.critical)
        healthy = critical_ok

        if self.report_path.exists():
            prev = json.loads(self.report_path.read_text(encoding="utf-8"))
            if prev.get("healthy") and prev.get("fingerprint") == fp:
                prev["generated_at"] = time.time()
                prev["rerun"] = True
                self.report_path.write_text(json.dumps(prev, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
                report = InitReport(**{**prev, "probes": prev.get("probes", [])})
                if fail_loud and not report.healthy:
                    raise WorkbenchBroken(self._format_failures(probes))
                return report

        report = InitReport(
            healthy=healthy,
            fingerprint=fp,
            generated_at=time.time(),
            probes=[asdict(p) for p in probes],
            runtime={"python": sys.version.split()[0]},
            test_command=test_cmd,
            last_known_good_commit=commit,
            rerun=False,
        )
        self.report_path.write_text(json.dumps(report.to_dict(), ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
        if fail_loud and not healthy:
            raise WorkbenchBroken(self._format_failures(probes))
        return report

    @staticmethod
    def _format_failures(probes: list[ProbeResult]) -> str:
        bad = [f"{p.name}: {p.detail}" for p in probes if p.critical and not p.ok]
        return "init failed (workbench broken):\n- " + "\n- ".join(bad)

    def assert_healthy(self) -> InitReport:
        """
        Returns:
            report: 健康报告。
        Raises:
            WorkbenchBroken: 无报告或不健康。
        """
        if not self.report_path.exists():
            raise WorkbenchBroken("init_report.json missing; run init first")
        data = json.loads(self.report_path.read_text(encoding="utf-8"))
        if not data.get("healthy"):
            raise WorkbenchBroken("init_report unhealthy; refuse agent loop")
        return InitReport(**data)


def make_workbench(
    *,
    healthy: bool = True,
    with_api_key: bool = True,
    stale: bool = False,
    missing_test: bool = False,
) -> Path:
    """
    构造演示用迷你工作台。

    Args:
        healthy: 是否构造可通过关键探测的布局。
        with_api_key: 是否注入假 API key 到环境。
        stale: 是否标记状态过期。
        missing_test: 是否缺少 test_command。
    Returns:
        root: 临时仓库根。
    """
    root = Path(tempfile.mkdtemp(prefix="init_wb_"))
    (root / "src").mkdir()
    (root / "tests").mkdir()
    (root / "AGENTS.md").write_text("# router\n", encoding="utf-8")
    (root / "agent_state.json").write_text(
        json.dumps({"schema_version": 1, "stale": stale, "active_task_id": None}, ensure_ascii=False),
        encoding="utf-8",
    )
    (root / "task_board.json").write_text(
        json.dumps({"stale": stale, "tasks": []}, ensure_ascii=False),
        encoding="utf-8",
    )
    (root / "last_known_good.txt").write_text("abc1234deadbeef", encoding="utf-8")
    (root / "requirements.txt").write_text("# stdlib only demo\n", encoding="utf-8")
    wb: dict[str, Any] = {}
    if not missing_test:
        wb["test_command"] = "python -m pytest -q"
    (root / "workbench.json").write_text(json.dumps(wb, ensure_ascii=False, indent=2), encoding="utf-8")
    if with_api_key:
        os.environ["DEEPSEEK_API_KEY"] = os.environ.get("DEEPSEEK_API_KEY") or "demo-key-for-init"
    elif "DEEPSEEK_API_KEY" in os.environ and not healthy:
        # 调用方可自行 pop；此处仅用于 healthy=False 场景提示
        pass
    if not healthy and missing_test:
        pass
    return root


print("initialization scripts ready | probes + init_report + idempotent rerun")


## 2. 玩具示例

健康门闩、幂等二次跑、缺 test_command / stale 状态 fail loud。


In [ ]:
def demo_init_scripts() -> None:
    """健康启动、幂等重跑、损坏工作台 fail-loud。"""
    saved_key = os.environ.get("DEEPSEEK_API_KEY")

    root = make_workbench(healthy=True, with_api_key=True)
    try:
        init = InitScript(root)
        r1 = init.run()
        assert r1.healthy and not r1.rerun
        assert init.report_path.exists()
        t1 = r1.generated_at

        time.sleep(0.02)
        r2 = init.run()
        assert r2.healthy and r2.rerun
        assert r2.fingerprint == r1.fingerprint
        assert r2.generated_at >= t1
        # 第二次除时间戳/rerun 外内容一致
        d1 = json.loads(init.report_path.read_text(encoding="utf-8"))
        assert d1["rerun"] is True
        print("idempotent rerun ok")

        init.assert_healthy()
        print("healthy gate ok")
    finally:
        shutil.rmtree(root, ignore_errors=True)

    # 缺测试命令 → 大声失败，拒绝进入 loop
    broken = make_workbench(missing_test=True, with_api_key=True)
    try:
        init_b = InitScript(broken)
        try:
            init_b.run(fail_loud=True)
            raise AssertionError("expected WorkbenchBroken")
        except WorkbenchBroken as e:
            assert "test_command" in str(e)
            print("fail loud on missing test_command ok")
        # 报告仍写入，但 healthy=false
        raw = json.loads(init_b.report_path.read_text(encoding="utf-8"))
        assert raw["healthy"] is False
        try:
            init_b.assert_healthy()
            raise AssertionError("expected unhealthy refuse")
        except WorkbenchBroken:
            print("refuse agent loop on unhealthy report ok")
    finally:
        shutil.rmtree(broken, ignore_errors=True)

    # 崩溃残留 stale
    stale_root = make_workbench(stale=True, with_api_key=True)
    try:
        init_s = InitScript(stale_root)
        try:
            init_s.run()
            raise AssertionError("expected stale failure")
        except WorkbenchBroken as e:
            assert "state_fresh" in str(e) or "stale" in str(e)
            print("stale state fail loud ok")
    finally:
        shutil.rmtree(stale_root, ignore_errors=True)

    if saved_key is not None:
        os.environ["DEEPSEEK_API_KEY"] = saved_key
    print("TOY DEMO OK")


demo_init_scripts()


## 3. 生产级：LangChain 工具驱动初始化 + DeepSeek

工具：`run_init` / `read_init_report` / `assert_healthy` / `enter_agent_loop`。未健康不得进入 loop。需 `DEEPSEEK_API_KEY`。


In [ ]:
import json
import os
import shutil
import sys
import tempfile
from pathlib import Path
from typing import Any

from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain_core.messages import AIMessage, BaseMessage, HumanMessage, ToolMessage
from langchain_core.tools import StructuredTool
from pydantic import BaseModel, Field

sys.path.append(str(Path("../../00_Common").resolve()))
from user_tools import load_project_env  # noqa: E402

load_project_env()

MODEL = "deepseek:deepseek-v4-flash"
PROD_ROOT: Path | None = None
PROD_INIT: InitScript | None = None
AGENT_LOOP_ALLOWED = False


def get_llm(*, temperature: float = 0.0) -> Any:
    """
    Returns:
        llm: DeepSeek chat model。
    """
    if not os.getenv("DEEPSEEK_API_KEY"):
        raise RuntimeError("DEEPSEEK_API_KEY missing; copy .env.example → .env")
    return init_chat_model(
        MODEL,
        temperature=temperature,
        extra_body={"thinking": {"type": "disabled"}},
    )


def reset_prod_workbench(*, broken: bool = False) -> str:
    """
    重建演示工作台。

    Args:
        broken: True 时故意缺少 test_command。
    Returns:
        json: root 与 broken 标记。
    """
    global PROD_ROOT, PROD_INIT, AGENT_LOOP_ALLOWED
    if PROD_ROOT and PROD_ROOT.exists():
        shutil.rmtree(PROD_ROOT, ignore_errors=True)
    AGENT_LOOP_ALLOWED = False
    PROD_ROOT = make_workbench(missing_test=broken, with_api_key=True)
    # 生产探测用真实 env；ensure key from load_project_env
    PROD_INIT = InitScript(PROD_ROOT)
    return json.dumps({"root": str(PROD_ROOT), "broken": broken}, ensure_ascii=False)


def _init() -> InitScript:
    if PROD_INIT is None:
        reset_prod_workbench(broken=False)
    assert PROD_INIT is not None
    return PROD_INIT


class EmptyArgs(BaseModel):
    pass


class ResetArgs(BaseModel):
    broken: bool = Field(False, description="若 true，构造缺少 test_command 的损坏工作台")


class FailLoudArgs(BaseModel):
    fail_loud: bool = True


def build_init_tools() -> list[StructuredTool]:
    def _reset(**kwargs: Any) -> str:
        a = ResetArgs(**kwargs)
        return reset_prod_workbench(broken=a.broken)

    def _run_init(**kwargs: Any) -> str:
        global AGENT_LOOP_ALLOWED
        a = FailLoudArgs(**kwargs)
        try:
            report = _init().run(fail_loud=a.fail_loud)
            AGENT_LOOP_ALLOWED = bool(report.healthy)
            return json.dumps(report.to_dict(), ensure_ascii=False)
        except WorkbenchBroken as e:
            AGENT_LOOP_ALLOWED = False
            # 报告已写入
            payload = {"ok": False, "error": str(e)}
            if _init().report_path.exists():
                payload["report"] = json.loads(_init().report_path.read_text(encoding="utf-8"))
            return json.dumps(payload, ensure_ascii=False)

    def _read_report(**kwargs: Any) -> str:
        p = _init().report_path
        if not p.exists():
            return json.dumps({"ok": False, "error": "init_report.json missing"}, ensure_ascii=False)
        return p.read_text(encoding="utf-8")

    def _assert_healthy(**kwargs: Any) -> str:
        global AGENT_LOOP_ALLOWED
        try:
            report = _init().assert_healthy()
            AGENT_LOOP_ALLOWED = True
            return json.dumps({"ok": True, "report": report.to_dict()}, ensure_ascii=False)
        except WorkbenchBroken as e:
            AGENT_LOOP_ALLOWED = False
            return json.dumps({"ok": False, "error": str(e)}, ensure_ascii=False)

    def _enter_loop(**kwargs: Any) -> str:
        """只有 healthy 才允许进入 agent loop（演示门闩）。"""
        if not AGENT_LOOP_ALLOWED:
            try:
                _init().assert_healthy()
            except WorkbenchBroken as e:
                return json.dumps({"ok": False, "entered": False, "error": str(e)}, ensure_ascii=False)
        return json.dumps(
            {
                "ok": True,
                "entered": True,
                "message": "agent loop started; cold-start tax already paid via init_report.json",
            },
            ensure_ascii=False,
        )

    return [
        StructuredTool.from_function(name="reset_workbench", description="重建演示工作台。", func=_reset, args_schema=ResetArgs),
        StructuredTool.from_function(
            name="run_init",
            description="跑初始化脚本，写 init_report.json；失败则 fail loud。",
            func=_run_init,
            args_schema=FailLoudArgs,
        ),
        StructuredTool.from_function(name="read_init_report", description="读取 init_report.json。", func=_read_report, args_schema=EmptyArgs),
        StructuredTool.from_function(
            name="assert_healthy",
            description="确认报告健康，否则拒绝进入 loop。",
            func=_assert_healthy,
            args_schema=EmptyArgs,
        ),
        StructuredTool.from_function(
            name="enter_agent_loop",
            description="仅在初始化健康时进入 agent loop。",
            func=_enter_loop,
            args_schema=EmptyArgs,
        ),
    ]


INIT_TOOLS = build_init_tools()


def build_init_agent():
    """
    Returns:
        agent: 必须先 init 再进 loop 的 control agent。
    """
    system = (
        "你是工作台启动代理。每个冷启动会话必须先 run_init，读 init_report，"
        "healthy 才能 enter_agent_loop。损坏时 fail loud，不要猜、不要硬闯。\n"
        "幂等：可再跑一次 run_init，应看到 rerun=true。\n"
        "用中文简短说明。"
    )
    return create_agent(get_llm(), INIT_TOOLS, system_prompt=system)


def _print_trace(messages: list[BaseMessage], limit: int = 28) -> None:
    n = 0
    for m in messages:
        if n >= limit:
            break
        if isinstance(m, HumanMessage):
            print(f"USER: {m.content}")
            n += 1
        elif isinstance(m, AIMessage):
            if m.tool_calls:
                for tc in m.tool_calls:
                    print(f"ACTION: {tc['name']}({tc.get('args', {})})")
                    n += 1
            elif m.content:
                print(f"ASSISTANT: {str(m.content)[:280]}")
                n += 1
        elif isinstance(m, ToolMessage):
            print(f"OBS[{m.name}]: {str(m.content)[:240]}")
            n += 1


print(f"initialization production ready | {MODEL}")


## 4. 生产示例

无 `DEEPSEEK_API_KEY` 则跳过 LLM；脚本路径仍验证健康/损坏/幂等。


In [ ]:
def demo_prod_init() -> None:
    """脚本化门禁 +（有 key 时）DeepSeek 启动流程。"""
    # 脚本：健康路径
    reset_prod_workbench(broken=False)
    r = _init().run()
    assert r.healthy and not r.rerun
    r2 = _init().run()
    assert r2.rerun and r2.fingerprint == r.fingerprint
    _init().assert_healthy()
    print("=== scripted healthy ===")
    print(json.dumps({"healthy": True, "idempotent": True, "test_command": r.test_command}, ensure_ascii=False))

    # 脚本：损坏路径
    reset_prod_workbench(broken=True)
    try:
        _init().run(fail_loud=True)
        raise AssertionError("expected broken")
    except WorkbenchBroken:
        pass
    raw = json.loads(_init().report_path.read_text(encoding="utf-8"))
    assert raw["healthy"] is False
    print("=== scripted broken ===")
    print(json.dumps({"healthy": False, "fail_loud": True}, ensure_ascii=False))

    if not os.getenv("DEEPSEEK_API_KEY"):
        print("SKIP llm agent: DEEPSEEK_API_KEY missing")
        print("PROD DEMO OK (scripted only)")
        return

    agent = build_init_agent()
    prompt = (
        "reset_workbench broken=false，run_init，再 run_init 一次确认幂等 rerun，"
        "assert_healthy，enter_agent_loop；"
        "然后 reset_workbench broken=true，run_init（应失败），"
        "尝试 enter_agent_loop（应被拒），中文说明为何初始化要 fail loud。"
    )
    result = agent.invoke({"messages": [HumanMessage(content=prompt)]})
    print("=== control agent ===")
    _print_trace(list(result["messages"]))

    # 最终应停在 broken 工作台且未放行
    assert PROD_INIT is not None
    final = json.loads(PROD_INIT.report_path.read_text(encoding="utf-8"))
    assert final.get("healthy") is False
    assert AGENT_LOOP_ALLOWED is False
    print("PROD DEMO OK")


demo_prod_init()
